In [ ]:
import os
import sys
import difflib
import collections # Still needed for PdsNode.__repr__ potentially, and if any local helpers remain
import re 

# Import from pds_parser (ensure all node types are listed)
from pds_parser import (
    PdsParser, PdsBlock, PdsKeyValuePair, PdsList, 
    PdsComment, PdsBlankLine, PdsOperatorCondition, PdsNode
)
# Import from pds_differ
from pds_differ import PdsDiffer, PdsChange # PdsDiffer now handles simulation

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Paths (UNCHANGED) ---
SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"
INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"

# --- File I/O Helpers (UNCHANGED) ---
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f: return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f: return f.read()
        except Exception as e_inner: sys.stderr.write(f"ERROR reading {filepath} (fallback): {e_inner}\n"); return None
    except FileNotFoundError: sys.stderr.write(f"WARNING: File not found: {filepath}\n"); return None
    except Exception as e: sys.stderr.write(f"ERROR reading {filepath}: {e}\n"); return None

def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f: f.write(content); return True
    except Exception as e: sys.stderr.write(f"ERROR writing to {filepath}: {e}\n"); return False

# --- Tree Navigation Helpers ---
# These are now moved to pds_differ.py:
# _parse_indexed_identifier_for_nav
# _get_key_from_path_segment
# find_container_by_key_path
# find_node_by_old_path_in_sim_tree
# find_copied_node_recursive
# find_node_and_parent_in_sim_tree

def normalize_for_comparison(text):
    if not text: return ""
    # Normalize newlines: replace multiple blank lines with a single one. Also strip leading/trailing whitespace.
    return re.sub(r'\n(\s*\n)+', '\n', text).strip()

# g_subsumed_mod_block_paths - This is now managed within pds_differ.py

def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    # g_subsumed_mod_block_paths is managed by PdsDiffer during simulation now

    print(f"\n{'='*20} RUNNING TEST: {test_name} {'='*20}")
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved.")

    parser = PdsParser() 
    print(f"  Parsing Old file: {os.path.basename(old_filepath)}")
    old_nodes = parser.parse_file(old_filepath)
    print(f"  Parsing Mod file: {os.path.basename(mod_filepath)}")
    mod_nodes = parser.parse_file(mod_filepath)
    print(f"  Parsing New file: {os.path.basename(new_filepath)}")
    new_nodes = parser.parse_file(new_filepath)

    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []
    
    # Check if any content exists before proceeding
    if not (old_nodes or mod_nodes or new_nodes) and \
       not (old_content_raw or mod_content_raw or new_content_raw):
        print(f"SKIPPING TEST '{test_name}': All input files are empty or could not be parsed.")
        return

    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved.")
    
    normalized_old_raw = normalize_for_comparison(old_content_raw or "")
    normalized_mod_raw = normalize_for_comparison(mod_content_raw or "")
    normalized_new_raw = normalize_for_comparison(new_content_raw or "")

    if normalize_for_comparison(reconstructed_old) != normalized_old_raw: print(f"\nWARNING: Normalized reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}.")
    if normalize_for_comparison(reconstructed_mod) != normalized_mod_raw: print(f"\nWARNING: Normalized reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}.")
    if normalize_for_comparison(reconstructed_new) != normalized_new_raw: print(f"\nWARNING: Normalized reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}.")

    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)
    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes: print("    No significant changes detected by PdsDiffer.")
    for change in changes: print(change) # Print the PdsChange objects

    # _simulate_apply_change is now internal to PdsDiffer
    # Call the new method in PdsDiffer
    print(f"\n--- Test Script: Requesting PdsDiffer to SIMULATE MERGE for '{test_name}' ---")
    simulated_merged_nodes_root_list = differ.simulate_three_way_merge(changes, new_nodes)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes_root_list)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"\n  Simulated merged content saved by Test Script.")
    
    print(f"\n--- DIFF: SIMULATED MERGED vs NORMALIZED NEW VANILLA RAW for '{test_name}' ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    normalized_simulated_merged = normalize_for_comparison(simulated_merged_content)
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            normalized_new_raw.splitlines(keepends=True),
            normalized_simulated_merged.splitlines(keepends=True),
            fromfile='NORMALIZED_NEW_VANILLA_RAW', tofile='NORMALIZED_SIMULATED_MERGED', lineterm=''))
        if diff_lines: 
            f_diff.writelines(diff_lines)
            print(f"  Diff (Normalized) saved to: {os.path.basename(diff_filename)}")
        else: 
            print("  NORMALIZED SIMULATED MERGED is identical to NORMALIZED NEW VANILLA RAW.")
    print("-" * 80)

# --- Run Tests ---
print("Running diff tests with real CK3 files.")
# run_and_print_diff("Siege Events (real files)", SIEGE_EVENTS_OLD_VANILLA_PATH, SIEGE_EVENTS_MOD_PATH, SIEGE_EVENTS_NEW_VANILLA_PATH)
run_and_print_diff("00_tribal_innovations (real files)", INNOVATIONS_OLD_VANILLA_PATH, INNOVATIONS_MOD_PATH, INNOVATIONS_NEW_VANILLA_PATH)
print("\n--- All Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING TEST: 00_tribal_innovations (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\00_tribal_innovations_real_files
  Raw files saved.
  Parsing Old file: 00_tribal_innovations.txt
  Parsing Mod file: 00_tribal_innovations.txt
  Parsing New file: 00_tribal_innovations.txt
  Reconstructed files saved.




--- DETECTED CHANGES for '00_tribal_innovations (real files)' (35 changes) ---
PdsChange(Type='VANILLA_MODIFIED                             ', Path='innovation_bannus___0', 
          ParentCtx='ROOT_PARENT', 
          Nodes=[
            O:PdsBlock L83 I0 innovation_bannus={...},
            M:PdsBlock L83 I0 innovation_bannus={...},
     